# 🇳🇿 New Zealand Airborne Disease Data Scraper
**Source:** PHF Science (formerly ESR) — NZ Government Public Health Institute  
**Data:** Weekly Virology Reports (influenza A/B, RSV, COVID, rhinovirus) + Annual ARI Reports  
**Years:** 2019–2026  
**Output:** `NZ_Airborne_Disease_Data.xlsx` with one sheet per year + summary

---
### How to run
Run each cell **in order** from top to bottom.  
The scraper will skip files already downloaded, so it is safe to re-run.

## Cell 1 — Install libraries

In [1]:
import sys
!{sys.executable} -m pip install requests beautifulsoup4 pdfplumber pandas openpyxl --quiet
print('✓ All libraries installed')

✓ All libraries installed


## Cell 2 — Imports and configuration

In [2]:
import requests
from bs4 import BeautifulSoup
import pdfplumber
import pandas as pd
import os
import re
import time
from datetime import datetime

# ── Folders ────────────────────────────────────────────────────────────────
BASE_URL        = 'https://www.phfscience.nz'
VIROLOGY_DIR    = 'nz_data/virology_pdfs'
RESPIRATORY_DIR = 'nz_data/respiratory_reports'
OUTPUT_EXCEL    = 'NZ_Airborne_Disease_Data.xlsx'

os.makedirs(VIROLOGY_DIR,    exist_ok=True)
os.makedirs(RESPIRATORY_DIR, exist_ok=True)

HEADERS = {'User-Agent': 'Mozilla/5.0 (academic research)'}

print('✓ Configuration ready')
print(f'  Virology PDFs  → {VIROLOGY_DIR}/')
print(f'  Annual Reports → {RESPIRATORY_DIR}/')
print(f'  Final Excel    → {OUTPUT_EXCEL}')

✓ Configuration ready
  Virology PDFs  → nz_data/virology_pdfs/
  Annual Reports → nz_data/respiratory_reports/
  Final Excel    → NZ_Airborne_Disease_Data.xlsx


## Cell 3 — Helper functions

In [3]:
def fetch_page(url, retries=3):
    """Fetch a webpage, return None on 404 or repeated failure."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            if r.status_code == 200:
                return r
            elif r.status_code == 404:
                return None
        except requests.RequestException as e:
            print(f'  Retry {attempt+1}: {e}')
            time.sleep(2)
    return None


def download_file(file_url, dest_path):
    """Download a file to dest_path. Skip if already exists."""
    if os.path.exists(dest_path):
        return 'skipped'
    try:
        r = requests.get(file_url, headers=HEADERS, timeout=30, stream=True)
        if r.status_code == 200:
            with open(dest_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            return 'downloaded'
        else:
            return f'error_{r.status_code}'
    except Exception as e:
        return f'error: {e}'


def get_pdf_link_from_page(page_url):
    """Visit a report page and return the first PDF download URL found."""
    r = fetch_page(page_url)
    if not r:
        return None
    soup = BeautifulSoup(r.text, 'html.parser')
    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.pdf' in href.lower():
            return BASE_URL + href if href.startswith('/') else href
    return None

print('✓ Helper functions defined')

✓ Helper functions defined


## Cell 4 — Download weekly virology PDFs (2019–2026)
These are the primary airborne disease reports: influenza A/B subtypes, RSV, COVID, rhinovirus, adenovirus.  
URL pattern: `/digital-library/laboratory-based-virology-weekly-report-YYYY-week-WW/`

⏱ **This will take 10–20 minutes** depending on your internet speed.  
Progress is shown for every downloaded file. Already-downloaded files are skipped automatically.

In [4]:
current_year = datetime.now().year
current_week = datetime.now().isocalendar()[1]

downloaded_count = 0
skipped_count    = 0
missing_count    = 0

for year in range(2019, current_year + 1):
    max_week = current_week if year == current_year else 52
    print(f'\n── {year} (weeks 1–{max_week}) ──')

    for week in range(1, max_week + 1):
        week_str = f'{week:02d}'
        slug     = f'laboratory-based-virology-weekly-report-{year}-week-{week_str}'
        page_url = f'{BASE_URL}/digital-library/{slug}/'
        dest     = os.path.join(VIROLOGY_DIR, f'virology_{year}_week{week_str}.pdf')

        if os.path.exists(dest):
            skipped_count += 1
            continue

        pdf_url = get_pdf_link_from_page(page_url)

        if pdf_url is None:
            missing_count += 1
            continue   # week not published — normal for off-season years

        result = download_file(pdf_url, dest)
        if result == 'downloaded':
            print(f'  ↓ {year} Week {week_str}')
            downloaded_count += 1
        elif result == 'skipped':
            skipped_count += 1
        else:
            print(f'  ✗ {year} Week {week_str}: {result}')
            missing_count += 1

        time.sleep(0.8)  # polite delay

total_pdfs = len([f for f in os.listdir(VIROLOGY_DIR) if f.endswith('.pdf')])
print(f'\n━━━ Virology Download Complete ━━━')
print(f'  Downloaded : {downloaded_count}')
print(f'  Skipped    : {skipped_count} (already existed)')
print(f'  Not found  : {missing_count} (weeks not published, expected for 2020)')
print(f'  Total PDFs : {total_pdfs}')


── 2019 (weeks 1–52) ──

── 2020 (weeks 1–52) ──

── 2021 (weeks 1–52) ──

── 2022 (weeks 1–52) ──

── 2023 (weeks 1–52) ──

── 2024 (weeks 1–52) ──

── 2025 (weeks 1–52) ──

── 2026 (weeks 1–16) ──

━━━ Virology Download Complete ━━━
  Downloaded : 0
  Skipped    : 144 (already existed)
  Not found  : 236 (weeks not published, expected for 2020)
  Total PDFs : 144


## Cell 5 — Download annual ARI surveillance reports (2019–2024)
These contain full-year seasonal summaries with weekly breakdowns of influenza, RSV, COVID, and SARI hospitalisations.

In [5]:
annual_reports = [
    ('2019', '2019-acute-respiratory-illness-surveillance-report'),
    ('2021', '2021-acute-respiratory-illness-surveillance-report'),
    ('2022', '2022-acute-respiratory-illness-surveillance-report'),
    ('2023', '2023-acute-respiratory-illness-surveillance-report'),
    ('2024', '2024-acute-respiratory-illness-surveillance-report'),
]

for year, slug in annual_reports:
    page_url = f'{BASE_URL}/digital-library/{slug}/'
    dest     = os.path.join(RESPIRATORY_DIR, f'ari_annual_{year}.pdf')

    if os.path.exists(dest):
        print(f'  ✓ {year} already downloaded')
        continue

    pdf_url = get_pdf_link_from_page(page_url)
    if pdf_url:
        result = download_file(pdf_url, dest)
        print(f'  ↓ {year} ARI Report → {result}')
    else:
        print(f'  ✗ {year}: No PDF found at {page_url}')
    time.sleep(1)

print('\n✓ Annual report downloads complete')

  ✗ 2019: No PDF found at https://www.phfscience.nz/digital-library/2019-acute-respiratory-illness-surveillance-report/
  ✗ 2021: No PDF found at https://www.phfscience.nz/digital-library/2021-acute-respiratory-illness-surveillance-report/
  ✓ 2022 already downloaded
  ✓ 2023 already downloaded
  ✓ 2024 already downloaded

✓ Annual report downloads complete


## Cell 6 — Confirm what was downloaded

In [6]:
virology_files    = sorted([f for f in os.listdir(VIROLOGY_DIR) if f.endswith('.pdf')])
respiratory_files = sorted([f for f in os.listdir(RESPIRATORY_DIR) if f.endswith('.pdf')])

print(f'Virology PDFs downloaded: {len(virology_files)}')
print(f'Annual ARI reports:       {len(respiratory_files)}')

# Show year coverage
years_found = sorted(set(
    re.search(r'virology_(\d{4})_', f).group(1)
    for f in virology_files if re.search(r'virology_(\d{4})_', f)
))
print(f'\nYears covered: {years_found}')

# Show weekly coverage per year
print('\nWeeks per year:')
for yr in years_found:
    count = len([f for f in virology_files if f'virology_{yr}_' in f])
    print(f'  {yr}: {count} weeks')

Virology PDFs downloaded: 144
Annual ARI reports:       3

Years covered: ['2021', '2022', '2023', '2024', '2025', '2026']

Weeks per year:
  2021: 21 weeks
  2022: 3 weeks
  2023: 32 weeks
  2024: 40 weeks
  2025: 43 weeks
  2026: 5 weeks


## Cell 7 — Extract tables from virology PDFs
Each PDF contains tables with weekly counts per virus type per laboratory.  
This cell reads all tables and compiles them into a single DataFrame.

In [7]:
def extract_tables_from_pdf(pdf_path, year, week):
    """Extract all tables from a virology PDF and tag with year/week."""
    rows = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                tables = page.extract_tables()
                for table in tables:
                    if not table or len(table) < 2:
                        continue
                    header = [str(c).strip() if c else f'col_{i}'
                              for i, c in enumerate(table[0])]
                    for data_row in table[1:]:
                        if not data_row or all(c is None for c in data_row):
                            continue
                        row_dict = {
                            'year':        year,
                            'epi_week':    week,
                            'page':        page_num + 1,
                            'source_file': os.path.basename(pdf_path),
                        }
                        for col, val in zip(header, data_row):
                            row_dict[col] = str(val).strip() if val else ''
                        rows.append(row_dict)
    except Exception as e:
        print(f'  ✗ Error reading {pdf_path}: {e}')
    return rows


# ── Run extraction ─────────────────────────────────────────────────────────
pdf_files = sorted([f for f in os.listdir(VIROLOGY_DIR) if f.endswith('.pdf')])
print(f'Extracting tables from {len(pdf_files)} PDFs...\n')

all_rows = []
errors   = []

for i, fname in enumerate(pdf_files):
    match = re.search(r'virology_(\d{4})_week(\d+)', fname)
    if not match:
        continue
    year = int(match.group(1))
    week = int(match.group(2))

    pdf_path = os.path.join(VIROLOGY_DIR, fname)
    rows = extract_tables_from_pdf(pdf_path, year, week)
    all_rows.extend(rows)

    if (i + 1) % 20 == 0:
        print(f'  Processed {i+1}/{len(pdf_files)} PDFs — {len(all_rows)} rows so far')

print(f'\n✓ Extraction complete: {len(all_rows)} total rows from {len(pdf_files)} PDFs')

Extracting tables from 144 PDFs...

  Processed 20/144 PDFs — 168 rows so far
  Processed 40/144 PDFs — 614 rows so far
  Processed 60/144 PDFs — 1145 rows so far
  Processed 80/144 PDFs — 1705 rows so far
  Processed 100/144 PDFs — 2265 rows so far
  Processed 120/144 PDFs — 3140 rows so far
  Processed 140/144 PDFs — 4203 rows so far

✓ Extraction complete: 4447 total rows from 144 PDFs


## Cell 8 — Preview extracted data

In [8]:
if not all_rows:
    print('No rows extracted — PDFs may use image-based tables.')
    print('Try opening one PDF manually to inspect its format.')
else:
    df_raw = pd.DataFrame(all_rows)
    df_raw = df_raw.dropna(how='all')
    df_raw = df_raw[df_raw.apply(lambda r: any(str(v).strip() for v in r), axis=1)]

    print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
    print(f'Columns: {list(df_raw.columns)}')
    print(f'Years: {sorted(df_raw["year"].unique())}')
    print()
    display(df_raw.head(10))

Shape: 4447 rows × 40 columns
Columns: ['year', 'epi_week', 'page', 'source_file', 'Influenza viruses', 'Week 27', 'Total', 'Week 28', 'Week 29', 'Week 30', 'Week 31', 'Week 32', 'Week 33', 'Week 34', 'Week 35', 'Week 36', 'Week 37', 'Week 38', 'Week 39', 'Non-influenza respiratory viruses', 'Week 18', 'col_1', 'col_2', 'col_4', 'col_5', 'Tables and Figures below show the weekly influenza and common non-influenza viruses reported from the', 'Tables and Figures below show the weekly influenza and common non-influenza viruses reported from the laboratory', 'col_0', 'Month', 'col_3', 'col_6', 'col_7', 'col_8', 'col_9', 'col_10', 'col_11', 'col_12', 'RSV type', 'Enterovirus type', 'col_13']
Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]



,year,epi_week,page,source_file,Influenza viruses,Week 27,Total,Week 28,Week 29,Week 30,...,col_6,col_7,col_8,col_9,col_10,col_11,col_12,RSV type,Enterovirus type,col_13
0,2021,27,1,virology_2021_week27.pdf,No. of positive specimens,0,2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,27,1,virology_2021_week27.pdf,Influenza A,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021,27,1,virology_2021_week27.pdf,A (not subtyped),0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021,27,1,virology_2021_week27.pdf,A(H1N1)pdm09,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,27,1,virology_2021_week27.pdf,A(H1N1)pdm09 by PCR\nA/Victoria/2570/2019 (H1N...,0\n0,0\n0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2021,27,1,virology_2021_week27.pdf,A(H3N2),0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2021,27,1,virology_2021_week27.pdf,A(H3N2) by PCR,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2021,27,1,virology_2021_week27.pdf,A/Hong Kong/2671/2019 (H3N2)-like,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2021,27,1,virology_2021_week27.pdf,Influenza B,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2021,27,1,virology_2021_week27.pdf,B (lineage not determined),0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cell 9 — Save to Excel
Creates a multi-sheet Excel file:
- **All Years** — complete combined dataset
- **2019, 2021, 2022, 2023, 2024, 2025, 2026** — one sheet per year
- **Summary** — row count per year/week (quick quality check)

In [9]:
if 'df_raw' not in dir() or df_raw.empty:
    print('No data to save. Run Cell 7 first.')
else:
    print(f'Saving to {OUTPUT_EXCEL}...')

    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

        # Sheet 1: All data
        df_raw.to_excel(writer, sheet_name='All Years', index=False)
        print(f'  ✓ Sheet "All Years": {len(df_raw)} rows')

        # One sheet per year
        for year in sorted(df_raw['year'].unique()):
            year_df = df_raw[df_raw['year'] == year].reset_index(drop=True)
            year_df.to_excel(writer, sheet_name=str(year), index=False)
            print(f'  ✓ Sheet "{year}": {len(year_df)} rows')

        # Summary sheet
        summary = (df_raw.groupby(['year', 'epi_week'])
                         .size()
                         .reset_index(name='rows_extracted'))
        summary.to_excel(writer, sheet_name='Summary', index=False)
        print(f'  ✓ Sheet "Summary": {len(summary)} year/week combinations')

    print(f'\n✅ Done! File saved: {OUTPUT_EXCEL}')
    print(f'   You can find it in your file browser on the left panel.')

Saving to NZ_Airborne_Disease_Data.xlsx...
  ✓ Sheet "All Years": 4447 rows
  ✓ Sheet "2021": 182 rows
  ✓ Sheet "2022": 28 rows
  ✓ Sheet "2023": 823 rows
  ✓ Sheet "2024": 1120 rows
  ✓ Sheet "2025": 1991 rows
  ✓ Sheet "2026": 303 rows
  ✓ Sheet "Summary": 136 year/week combinations

✅ Done! File saved: NZ_Airborne_Disease_Data.xlsx
   You can find it in your file browser on the left panel.


## Cell 10 — Quick data quality check

In [10]:
if 'df_raw' in dir() and not df_raw.empty:
    print('=== DATA QUALITY SUMMARY ===')
    print(f'Total rows     : {len(df_raw):,}')
    print(f'Total columns  : {df_raw.shape[1]}')
    print(f'Years covered  : {sorted(df_raw["year"].unique())}')
    print(f'\nRows per year:')
    print(df_raw.groupby('year').size().to_string())
    print(f'\nMissing values per column:')
    missing = df_raw.isnull().sum()
    print(missing[missing > 0].to_string() if missing.any() else '  None — clean!')
    print(f'\nSample of first 5 rows:')
    display(df_raw.head())

=== DATA QUALITY SUMMARY ===
Total rows     : 4,447
Total columns  : 40
Years covered  : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Rows per year:
year
2021     182
2022      28
2023     823
2024    1120
2025    1991
2026     303

Missing values per column:
Influenza viruses                                                                                                   2347
Week 27                                                                                                             4433
Total                                                                                                               1119
Week 28                                                                                                             4433
Week 29                                                                                                             4433
Week 30                                                                           

,year,epi_week,page,source_file,Influenza viruses,Week 27,Total,Week 28,Week 29,Week 30,...,col_6,col_7,col_8,col_9,col_10,col_11,col_12,RSV type,Enterovirus type,col_13
0,2021,27,1,virology_2021_week27.pdf,No. of positive specimens,0,2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021,27,1,virology_2021_week27.pdf,Influenza A,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021,27,1,virology_2021_week27.pdf,A (not subtyped),0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021,27,1,virology_2021_week27.pdf,A(H1N1)pdm09,0,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021,27,1,virology_2021_week27.pdf,A(H1N1)pdm09 by PCR\nA/Victoria/2570/2019 (H1N...,0\n0,0\n0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## ✅ Done!

Your NZ airborne disease data is saved as **`NZ_Airborne_Disease_Data.xlsx`** in the same folder as this notebook.

**What the data contains:**
- Weekly counts of influenza A (H1N1, H3N2), influenza B, RSV, SARS-CoV-2, rhinovirus, adenovirus
- Lab-confirmed cases from the NZ national virus laboratory network (~70% NZ population coverage)
- 2019–2026 (2020 has very few weeks due to COVID lockdown suppression of respiratory viruses)

**Next steps:**
1. Open the Excel file and inspect column names from the PDFs
2. Standardize epi-week format to match your Australia dataset (ISO 8601)
3. Merge into your multi-country pipeline

**Data citation:**  
PHF Science (New Zealand Institute for Public Health and Forensic Science), Laboratory-based virology weekly reports. https://www.phfscience.nz

In [11]:
import pandas as pd
import numpy as np
import re

# ── Reshape: extract only the current week's data from each row ──────────────
reshaped_rows = []

for _, row in df_raw.iterrows():
    year     = row['year']
    epi_week = row['epi_week']
    virus    = str(row.get('Influenza viruses', '')).strip()

    if not virus or virus in ['nan', '']:
        continue

    # The current week's count column is named "Week {epi_week}"
    week_col = f'Week {epi_week}'
    count    = row.get(week_col, np.nan)

    # Also try "Total" column as fallback
    if pd.isna(count) or str(count).strip() in ['', 'nan']:
        count = row.get('Total', np.nan)

    # Clean count value — remove newlines, take first number
    count_str = str(count).split('\\n')[0].strip()
    try:
        count_clean = float(count_str)
    except:
        count_clean = np.nan

    reshaped_rows.append({
        'country':    'New Zealand',
        'year':       year,
        'epi_week':   epi_week,
        'virus_type': virus,
        'count':      count_clean,
        'source_file': row['source_file']
    })

df_clean = pd.DataFrame(reshaped_rows)

# Drop rows where virus_type is just a header repeated or irrelevant
exclude = ['no. of positive specimens', 'influenza viruses',
           'non-influenza respiratory viruses', 'virus']
df_clean = df_clean[
    ~df_clean['virus_type'].str.lower().str.strip().isin(exclude)
]
df_clean = df_clean.dropna(subset=['count'])
df_clean = df_clean[df_clean['count'] >= 0]

print(f"Reshaped: {len(df_clean)} rows")
print(f"Years: {sorted(df_clean['year'].unique())}")
print(f"Virus types found:")
for v in sorted(df_clean['virus_type'].unique()):
    print(f"  {v}")
display(df_clean.head(10))

Reshaped: 374 rows
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Virus types found:
  A (not subtyped)
  A Quadrivalent-Rapid Antigen Test (RAT)
  A(H1N1)pdm09
  A(H1N1)pdm09 by PCR
  A(H3N2)
  A(H3N2) by PCR
  A/Croatia/10136RV/2023 (H3N2)-like
  A/Darwin/6/2021 (H3N2)-like
  A/Hong Kong/2671/2019 (H3N2)-like
  A/Missouri/11/2025 (H1N1)pdm09-like virus
  A/Thailand/8/2022 (H3N2)-like
  A/Victoria/4897/2022 (H1N1)pdm09-like
  B (lineage not determined)
  B Quadrivalent-Rapid Antigen Test (RAT)
  B/Phuket/3073/2013-like
  B/Victoria lineage
  B/Yamagata lineage
  B/Yamagata lineage by PCR
  Influenza A
  Influenza B


,country,year,epi_week,virus_type,count,source_file
1,New Zealand,2021,27,Influenza A,0.0,virology_2021_week27.pdf
2,New Zealand,2021,27,A (not subtyped),0.0,virology_2021_week27.pdf
3,New Zealand,2021,27,A(H1N1)pdm09,0.0,virology_2021_week27.pdf
5,New Zealand,2021,27,A(H3N2),0.0,virology_2021_week27.pdf
6,New Zealand,2021,27,A(H3N2) by PCR,0.0,virology_2021_week27.pdf
7,New Zealand,2021,27,A/Hong Kong/2671/2019 (H3N2)-like,0.0,virology_2021_week27.pdf
8,New Zealand,2021,27,Influenza B,0.0,virology_2021_week27.pdf
9,New Zealand,2021,27,B (lineage not determined),0.0,virology_2021_week27.pdf
10,New Zealand,2021,27,B/Yamagata lineage,0.0,virology_2021_week27.pdf
12,New Zealand,2021,27,B/Victoria lineage,0.0,virology_2021_week27.pdf


In [12]:
# 2022 reports used a slightly different URL format — try alternative slugs
import time

print("Attempting 2022 downloads with alternative URL formats...")
fixed = 0

for week in range(1, 53):
    week_str = f'{week:02d}'
    dest = f'nz_data/virology_pdfs/virology_2022_week{week_str}.pdf'
    
    if os.path.exists(dest):
        continue
    
    # Try alternative slug formats PHF Science used in 2022
    alt_slugs = [
        f'laboratory-based-virology-weekly-report-2022-week-{week_str}',
        f'virology-weekly-report-2022-week-{week_str}',
        f'laboratory-based-virology-report-2022-week-{week_str}',
    ]
    
    for slug in alt_slugs:
        page_url = f'{BASE_URL}/digital-library/{slug}/'
        pdf_url  = get_pdf_link_from_page(page_url)
        if pdf_url:
            result = download_file(pdf_url, dest)
            if result == 'downloaded':
                print(f'  ↓ 2022 Week {week_str} (via {slug})')
                fixed += 1
            break
    time.sleep(0.5)

print(f'\n✓ Fixed: {fixed} additional 2022 files downloaded')

Attempting 2022 downloads with alternative URL formats...

✓ Fixed: 0 additional 2022 files downloaded


In [15]:
# Open one PDF manually and print ALL table contents raw
import pdfplumber

# Pick one PDF that should have data (winter season = weeks 19-40)
test_pdf = 'nz_data/virology_pdfs/virology_2023_week25.pdf'

with pdfplumber.open(test_pdf) as pdf:
    for i, page in enumerate(pdf.pages):
        print(f'\n=== PAGE {i+1} ===')
        tables = page.extract_tables()
        for j, table in enumerate(tables):
            print(f'\n--- Table {j+1} ---')
            for row in table:
                print(row)


=== PAGE 1 ===

--- Table 1 ---
['Influenza viruses', None, None, 'Total', None, None]
['No. of positive specimens', None, None, '5214', None, None]
['', 'Influenza A', '', '', '3397', '']
['', 'A (not subtyped)', '', '', '2780', '']
['', 'A(H1N1)pdm09', '', '', '567', '']
['', 'A(H1N1)pdm09 by PCR', '', '', '475', '']
['', 'A/Sydney/5/2021 (H1N1)pdm09-like', '', '', '92', '']
['', 'A(H3N2)', '', '', '50', '']
['', 'A(H3N2) by PCR', '', '', '50', '']
['A/Darwin/6/2021 (H3N2)-like', None, None, '0', None, None]
['', 'Influenza B', '', '', '1817', '']
['', 'B (lineage not determined)', '', '', '1580', '']
['', 'B/Yamagata lineage', '', '', '0', '']
['', 'B/Yamagata lineage by PCR', '', '', '0', '']
['', 'B/Phuket/3073/2013-like', '', '', '0', '']
['', 'B/Victoria lineage', '', '', '237', '']
['B/Victoria lineage by PCR\nB/Austria/1359417/2021-like', None, None, '167\n70', None, None]

--- Table 2 ---
['Non-influenza respiratory viruses', None, None, 'Total', None, None]
['No. of positiv

In [16]:
# 2022 reports were on esr.cri.nz (old domain) before PHF Science rebrand
ESR_BASE = 'https://www.esr.cri.nz'
fixed = 0

for week in range(1, 53):
    week_str = f'{week:02d}'
    dest = f'nz_data/virology_pdfs/virology_2022_week{week_str}.pdf'
    
    if os.path.exists(dest):
        continue

    # Try old ESR domain slugs
    old_slugs = [
        f'/digital-library/laboratory-based-virology-weekly-report-2022-week-{week_str}/',
        f'/digital-library/laboratory-based-virology-weekly-report-2022-week-{week}/',
    ]
    
    for slug in old_slugs:
        page_url = ESR_BASE + slug
        pdf_url  = get_pdf_link_from_page(page_url)
        if pdf_url:
            result = download_file(pdf_url, dest)
            if result == 'downloaded':
                print(f'  ↓ 2022 Week {week_str}')
                fixed += 1
            break
    time.sleep(0.5)

print(f'\n✓ 2022 files downloaded: {fixed}')


✓ 2022 files downloaded: 0


In [17]:
all_rows_fixed = []

for fname in sorted([f for f in os.listdir(VIROLOGY_DIR) if f.endswith('.pdf')]):
    match = re.search(r'virology_(\d{4})_week(\d+)', fname)
    if not match:
        continue
    year = int(match.group(1))
    week = int(match.group(2))
    pdf_path = os.path.join(VIROLOGY_DIR, fname)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    if not table or len(table) < 2:
                        continue

                    # Determine table category from header row
                    header_text = str(table[0][0]).strip() if table[0][0] else ''
                    if 'Influenza' in header_text:
                        category = 'Influenza'
                    elif 'Non-influenza' in header_text or 'respiratory' in header_text.lower():
                        category = 'Non-influenza respiratory'
                    else:
                        category = 'Other'

                    for row in table[1:]:
                        if not row or len(row) < 4:
                            continue

                        # Virus name is always index 1 (index 0 is blank/category)
                        virus = str(row[1]).strip() if row[1] else ''
                        if not virus or virus.lower() in ['', 'nan', 'none']:
                            # Try index 0 if index 1 is empty
                            virus = str(row[0]).strip() if row[0] else ''

                        if not virus or virus.lower() in ['', 'nan', 'none',
                                                           'no. of positive specimens']:
                            continue

                        # Count is always at index 3 (Total column)
                        count_raw = str(row[3]).strip() if len(row) > 3 and row[3] else '0'
                        # Clean: take first number, handle '167\n70' style
                        count_clean = count_raw.split('\n')[0].strip()
                        try:
                            count_val = float(count_clean)
                        except:
                            continue

                        all_rows_fixed.append({
                            'country':   'New Zealand',
                            'year':       year,
                            'epi_week':   week,
                            'category':   category,
                            'virus_type': virus,
                            'count':      count_val,
                            'source_file': fname
                        })
    except Exception as e:
        print(f'Error reading {fname}: {e}')

df_fixed = pd.DataFrame(all_rows_fixed)

# Exclude header-repeat rows
exclude = ['influenza viruses', 'non-influenza respiratory viruses',
           'no. of positive specimens', 'virus']
df_fixed = df_fixed[~df_fixed['virus_type'].str.lower().str.strip().isin(exclude)]
df_fixed = df_fixed[df_fixed['count'] >= 0]

print(f'✓ Extracted: {len(df_fixed)} rows')
print(f'Years: {sorted(df_fixed["year"].unique())}')
print(f'\nSample counts (should NOT be all zeros now):')
display(df_fixed[df_fixed['count'] > 0].head(10))

✓ Extracted: 3562 rows
Years: [np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Sample counts (should NOT be all zeros now):


,country,year,epi_week,category,virus_type,count,source_file
14,New Zealand,2023,17,Influenza,B/Victoria lineage by PCR\nB/Austria/1359417/2...,42.0,virology_2023_week17.pdf
16,New Zealand,2023,17,Non-influenza respiratory,Respiratory syncytial virus (RSV),613.0,virology_2023_week17.pdf
18,New Zealand,2023,17,Non-influenza respiratory,Parainfluenza 2 (PIV2),48.0,virology_2023_week17.pdf
20,New Zealand,2023,17,Non-influenza respiratory,Rhinovirus (RV),722.0,virology_2023_week17.pdf
22,New Zealand,2023,17,Non-influenza respiratory,Human metapneumovirus (hMPV),63.0,virology_2023_week17.pdf
38,New Zealand,2023,18,Influenza,B/Victoria lineage by PCR\nB/Austria/1359417/2...,22.0,virology_2023_week18.pdf
40,New Zealand,2023,18,Non-influenza respiratory,Respiratory syncytial virus (RSV),1.0,virology_2023_week18.pdf
42,New Zealand,2023,18,Non-influenza respiratory,Parainfluenza 2 (PIV2),11.0,virology_2023_week18.pdf
44,New Zealand,2023,18,Non-influenza respiratory,Rhinovirus (RV),399.0,virology_2023_week18.pdf
46,New Zealand,2023,18,Non-influenza respiratory,Human metapneumovirus (hMPV),102.0,virology_2023_week18.pdf


In [18]:
OUTPUT_EXCEL = 'NZ_Airborne_Disease_Data.xlsx'

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    df_fixed.to_excel(writer, sheet_name='All Years', index=False)
    for year in sorted(df_fixed['year'].unique()):
        yr_df = df_fixed[df_fixed['year'] == year].reset_index(drop=True)
        yr_df.to_excel(writer, sheet_name=str(year), index=False)
        print(f"  ✓ Sheet '{year}': {len(yr_df)} rows")
    pivot = df_fixed.pivot_table(
        index=['year', 'epi_week'],
        columns='virus_type',
        values='count',
        aggfunc='sum'
    ).reset_index()
    pivot.to_excel(writer, sheet_name='Pivot_Weekly', index=False)
    summary = (df_fixed.groupby(['year','epi_week','virus_type'])['count']
                       .sum().reset_index())
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f'\n✅ Saved: {OUTPUT_EXCEL}')

  ✓ Sheet '2023': 624 rows
  ✓ Sheet '2024': 960 rows
  ✓ Sheet '2025': 1710 rows
  ✓ Sheet '2026': 268 rows

✅ Saved: NZ_Airborne_Disease_Data.xlsx


In [19]:
# Check what PDFs actually exist per year
import os, re

virology_files = os.listdir('nz_data/virology_pdfs')
years_found = {}
for f in virology_files:
    match = re.search(r'virology_(\d{4})_week(\d+)', f)
    if match:
        yr = match.group(1)
        years_found[yr] = years_found.get(yr, 0) + 1

print("PDFs on disk per year:")
for yr in sorted(years_found):
    print(f"  {yr}: {years_found[yr]} PDFs")

# Check if 2021 PDFs exist but weren't processed
files_2021 = [f for f in virology_files if '2021' in f]
print(f"\n2021 files found: {len(files_2021)}")
if files_2021:
    print("Sample:", files_2021[:3])

PDFs on disk per year:
  2021: 21 PDFs
  2022: 3 PDFs
  2023: 32 PDFs
  2024: 40 PDFs
  2025: 43 PDFs
  2026: 5 PDFs

2021 files found: 21
Sample: ['virology_2021_week38.pdf', 'virology_2021_week29.pdf', 'virology_2021_week32.pdf']


In [20]:
all_rows_v2 = []

for fname in sorted([f for f in os.listdir(VIROLOGY_DIR) if f.endswith('.pdf')]):
    match = re.search(r'virology_(\d{4})_week(\d+)', fname)
    if not match:
        continue
    year = int(match.group(1))
    week = int(match.group(2))
    pdf_path = os.path.join(VIROLOGY_DIR, fname)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    if not table or len(table) < 2:
                        continue

                    # Detect table category
                    header_text = str(table[0][0]).strip() if table[0][0] else ''
                    if 'Influenza' in header_text:
                        category = 'Influenza'
                    elif 'Non-influenza' in header_text or 'respiratory' in header_text.lower():
                        category = 'Non-influenza respiratory'
                    else:
                        category = 'Other'

                    # ── Detect table format ──────────────────────────────
                    # Format A (2023+): columns = [virus, '', '', Total, ...]
                    # Format B (2021):  columns = [virus, '', '', Week27, Total, ...]
                    header_row = [str(c).strip() if c else '' for c in table[0]]
                    
                    # Find Total column index
                    total_idx = None
                    for i, h in enumerate(header_row):
                        if h.lower() == 'total':
                            total_idx = i
                            break
                    
                    # Find current week column index (for Format B)
                    week_col_str = f'Week {week}'
                    week_idx = None
                    for i, h in enumerate(header_row):
                        if h.strip() == week_col_str:
                            week_idx = i
                            break

                    for row in table[1:]:
                        if not row or len(row) < 2:
                            continue

                        # Get virus name — try index 1 first, then 0
                        virus = str(row[1]).strip() if row[1] else ''
                        if not virus or virus.lower() in ['', 'nan', 'none']:
                            virus = str(row[0]).strip() if row[0] else ''
                        if not virus or virus.lower() in ['', 'nan', 'none',
                                                           'no. of positive specimens']:
                            continue

                        # Get count — prefer current week, fallback to Total
                        count_raw = None
                        if week_idx and len(row) > week_idx:
                            count_raw = str(row[week_idx]).strip()
                        if not count_raw or count_raw in ['', 'nan', 'None']:
                            if total_idx and len(row) > total_idx:
                                count_raw = str(row[total_idx]).strip()
                        if not count_raw or count_raw in ['', 'nan', 'None']:
                            # Last resort: index 3
                            count_raw = str(row[3]).strip() if len(row) > 3 else '0'

                        # Clean count
                        count_clean = count_raw.split('\n')[0].strip()
                        try:
                            count_val = float(count_clean)
                        except:
                            continue

                        all_rows_v2.append({
                            'country':    'New Zealand',
                            'year':        year,
                            'epi_week':    week,
                            'category':    category,
                            'virus_type':  virus,
                            'count':       count_val,
                            'source_file': fname
                        })
    except Exception as e:
        print(f'Error: {fname}: {e}')

df_v2 = pd.DataFrame(all_rows_v2)
exclude = ['influenza viruses', 'non-influenza respiratory viruses',
           'no. of positive specimens', 'virus']
df_v2 = df_v2[~df_v2['virus_type'].str.lower().str.strip().isin(exclude)]
df_v2 = df_v2[df_v2['count'] >= 0]

print(f'✓ Total rows: {len(df_v2)}')
print(f'Years: {sorted(df_v2["year"].unique())}')
print(f'\nRows per year:')
print(df_v2.groupby('year').size().to_string())
print(f'\nNon-zero rows: {len(df_v2[df_v2["count"] > 0])}')
display(df_v2[df_v2['count'] > 0].head(10))

✓ Total rows: 1385
Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

Rows per year:
year
2021    182
2022     27
2023    300
2024    240
2025    491
2026    145

Non-zero rows: 1090


,country,year,epi_week,category,virus_type,count,source_file
28,New Zealand,2021,29,Influenza,1,1.0,virology_2021_week29.pdf
33,New Zealand,2021,29,Influenza,1,1.0,virology_2021_week29.pdf
34,New Zealand,2021,29,Influenza,1,1.0,virology_2021_week29.pdf
42,New Zealand,2021,30,Influenza,1,1.0,virology_2021_week30.pdf
47,New Zealand,2021,30,Influenza,1,1.0,virology_2021_week30.pdf
48,New Zealand,2021,30,Influenza,1,1.0,virology_2021_week30.pdf
70,New Zealand,2021,32,Influenza,1,1.0,virology_2021_week32.pdf
71,New Zealand,2021,32,Influenza,1,1.0,virology_2021_week32.pdf
72,New Zealand,2021,32,Influenza,1,1.0,virology_2021_week32.pdf
182,New Zealand,2022,18,Non-influenza respiratory,176,176.0,virology_2022_week18.pdf


In [21]:
OUTPUT_EXCEL = 'NZ_Airborne_Disease_Data.xlsx'

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
    df_v2.to_excel(writer, sheet_name='All Years', index=False)
    for year in sorted(df_v2['year'].unique()):
        yr_df = df_v2[df_v2['year'] == year].reset_index(drop=True)
        yr_df.to_excel(writer, sheet_name=str(year), index=False)
        print(f"  ✓ Sheet '{year}': {len(yr_df)} rows")
    pivot = df_v2.pivot_table(
        index=['year', 'epi_week'],
        columns='virus_type',
        values='count',
        aggfunc='sum'
    ).reset_index()
    pivot.to_excel(writer, sheet_name='Pivot_Weekly', index=False)
    summary = (df_v2.groupby(['year','epi_week','virus_type'])['count']
                    .sum().reset_index())
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f'\n✅ Saved: {OUTPUT_EXCEL}')
print(f'Right-click → Download to open in Excel')

  ✓ Sheet '2021': 182 rows
  ✓ Sheet '2022': 27 rows
  ✓ Sheet '2023': 300 rows
  ✓ Sheet '2024': 240 rows
  ✓ Sheet '2025': 491 rows
  ✓ Sheet '2026': 145 rows

✅ Saved: NZ_Airborne_Disease_Data.xlsx
Right-click → Download to open in Excel


In [23]:
# Diagnose 2021 format specifically
test_pdf_2021 = 'nz_data/virology_pdfs/virology_2021_week29.pdf'

with pdfplumber.open(test_pdf_2021) as pdf:
    for i, page in enumerate(pdf.pages[:1]):
        tables = page.extract_tables()
        for j, table in enumerate(tables[:2]):
            print(f'\n--- Table {j+1} ---')
            print(f'Header row: {table[0]}')
            print(f'Row 2:      {table[1]}')
            print(f'Row 3:      {table[2] if len(table) > 2 else "N/A"}')
            print(f'Row 4:      {table[3] if len(table) > 3 else "N/A"}')


--- Table 1 ---
Header row: ['Influenza viruses', 'Week 29', 'Total']
Row 2:      ['No. of positive specimens', '1', '3']
Row 3:      ['Influenza A', '0', '1']
Row 4:      ['A (not subtyped)', '0', '0']


In [22]:
all_rows_final = []

for fname in sorted([f for f in os.listdir(VIROLOGY_DIR) if f.endswith('.pdf')]):
    match = re.search(r'virology_(\d{4})_week(\d+)', fname)
    if not match:
        continue
    year = int(match.group(1))
    week = int(match.group(2))
    pdf_path = os.path.join(VIROLOGY_DIR, fname)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    if not table or len(table) < 2:
                        continue

                    # Category from header
                    header_text = str(table[0][0]).strip() if table[0][0] else ''
                    if 'Influenza' in header_text:
                        category = 'Influenza'
                    elif 'Non-influenza' in header_text or 'respiratory' in header_text.lower():
                        category = 'Non-influenza respiratory'
                    else:
                        continue  # skip unrelated tables

                    header_row = [str(c).strip() if c else '' for c in table[0]]
                    n_cols = len(header_row)

                    # Find Total column
                    total_idx = next((i for i, h in enumerate(header_row)
                                      if h.lower() == 'total'), None)

                    for row in table[1:]:
                        if not row or len(row) < 2:
                            continue

                        # ── Get virus name ─────────────────────────────
                        # Try index 1 first (most formats), then index 0
                        virus = ''
                        for idx in [1, 0]:
                            val = str(row[idx]).strip() if len(row) > idx and row[idx] else ''
                            # Must be a real virus name: contains letters, not just numbers
                            if val and not val.replace('.','').replace('\n','').isdigit():
                                virus = val.split('\n')[0].strip()
                                break

                        if not virus:
                            continue

                        skip_terms = ['no. of positive specimens', 'influenza viruses',
                                      'non-influenza respiratory viruses', 'virus',
                                      'influenza a', 'influenza b']
                        if virus.lower() in skip_terms:
                            continue

                        # ── Get count ──────────────────────────────────
                        count_val = None

                        # Try Total column first
                        if total_idx is not None and len(row) > total_idx:
                            raw = str(row[total_idx]).split('\n')[0].strip()
                            try:
                                count_val = float(raw)
                            except:
                                pass

                        # Fallback: scan from right for first valid number
                        if count_val is None:
                            for cell in reversed(row):
                                raw = str(cell).split('\n')[0].strip() if cell else ''
                                try:
                                    count_val = float(raw)
                                    break
                                except:
                                    continue

                        if count_val is None or count_val < 0:
                            continue

                        all_rows_final.append({
                            'country':    'New Zealand',
                            'year':        year,
                            'epi_week':    week,
                            'category':    category,
                            'virus_type':  virus,
                            'count':       count_val,
                            'source_file': fname
                        })

    except Exception as e:
        print(f'Error: {fname}: {e}')

df_final = pd.DataFrame(all_rows_final)

print(f'✓ Total rows: {len(df_final)}')
print(f'Non-zero rows: {len(df_final[df_final["count"] > 0])}')
print(f'\nRows per year:')
print(df_final.groupby('year').size().to_string())
print(f'\nSample 2021 data:')
display(df_final[(df_final['year']==2021) & (df_final['count']>0)].head(5))
print(f'\nSample 2023 data:')
display(df_final[(df_final['year']==2023) & (df_final['count']>0)].head(5))

✓ Total rows: 2728
Non-zero rows: 2178

Rows per year:
year
2021    143
2022     24
2023    692
2024    880
2025    989

Sample 2021 data:


,country,year,epi_week,category,virus_type,count,source_file
9,New Zealand,2021,27,Influenza,B/Victoria lineage,2.0,virology_2021_week27.pdf
10,New Zealand,2021,27,Influenza,B/Victoria lineage by PCR,2.0,virology_2021_week27.pdf
20,New Zealand,2021,28,Influenza,B/Victoria lineage,2.0,virology_2021_week28.pdf
21,New Zealand,2021,28,Influenza,B/Victoria lineage by PCR,2.0,virology_2021_week28.pdf
25,New Zealand,2021,29,Influenza,A(H3N2),1.0,virology_2021_week29.pdf



Sample 2023 data:


,country,year,epi_week,category,virus_type,count,source_file
167,New Zealand,2023,11,Influenza,A (not subtyped),287.0,virology_2023_week11.pdf
168,New Zealand,2023,11,Influenza,A(H1N1)pdm09,156.0,virology_2023_week11.pdf
169,New Zealand,2023,11,Influenza,A(H1N1)pdm09 by PCR,156.0,virology_2023_week11.pdf
170,New Zealand,2023,11,Influenza,A(H3N2),23.0,virology_2023_week11.pdf
171,New Zealand,2023,11,Influenza,A(H3N2) by PCR,23.0,virology_2023_week11.pdf


In [24]:
OUTPUT_EXCEL = 'NZ_Airborne_Disease_Data.xlsx'

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:

    df_final.to_excel(writer, sheet_name='All Years', index=False)

    for year in sorted(df_final['year'].unique()):
        yr_df = df_final[df_final['year'] == year].reset_index(drop=True)
        yr_df.to_excel(writer, sheet_name=str(year), index=False)
        print(f"  ✓ Sheet '{year}': {len(yr_df)} rows")

    pivot = df_final.pivot_table(
        index=['year', 'epi_week'],
        columns='virus_type',
        values='count',
        aggfunc='sum'
    ).reset_index()
    pivot.to_excel(writer, sheet_name='Pivot_Weekly', index=False)

    summary = (df_final.groupby(['year','epi_week','virus_type'])['count']
                       .sum().reset_index())
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f'\n✅ Final file saved: {OUTPUT_EXCEL}')
print(f'Total rows: {len(df_final)} | Non-zero: {len(df_final[df_final["count"]>0])}')
print(f'Years: {sorted(df_final["year"].unique())}')
print(f'\nRight-click NZ_Airborne_Disease_Data.xlsx → Download')

  ✓ Sheet '2021': 143 rows
  ✓ Sheet '2022': 24 rows
  ✓ Sheet '2023': 692 rows
  ✓ Sheet '2024': 880 rows
  ✓ Sheet '2025': 989 rows

✅ Final file saved: NZ_Airborne_Disease_Data.xlsx
Total rows: 2728 | Non-zero: 2178
Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Right-click NZ_Airborne_Disease_Data.xlsx → Download
